# Parquet - Missing Records

Remove expected observations, scattered or across multiple periods; a known reporting schedule makes the gaps diagnosable.

**Instructions:** Run `prepare_data_parquet.ipynb` to create clean `data/weather.parquet` first (`uv add pyarrow`). This notebook overwrites the Parquet file; restore the clean artifact before rerunning. For combination, use only the required config/function/call, and load/save once.

In [ ]:
from pathlib import Path
from random import Random

import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

# Configuration
DATA_PATH = Path("../../data/weather.parquet")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive; local dates.
SEED = 814205

# Missing Records
KEY_COLUMNS = ["time"]
RECORD_PLACEMENT = "mixed"  # scattered, grouped, mixed, random
RECORD_RATE = (0.01, 0.03)  # Share of current eligible records removed.
RECORD_GROUPS = 3
RECORD_GROUPED_SHARE = 0.70
EXPECTED_FREQUENCY = "h"  # Only for known full hourly data; None for irregular events.

In [ ]:
# Shared selection: include once in an assembled dirty_data.py.
def scoped_rows(df, *, time_column, timezone, start_date, end_date):
    if time_column not in df:
        raise ValueError("Time column not found.")
    times = pd.to_datetime(df[time_column], utc=True, errors="raise")
    if times.isna().any():
        raise ValueError("Time column contains missing timestamps.")
    start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC")
    end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC")
    if start >= end:
        raise ValueError("Start date must precede end date.")
    scoped = df.loc[times.ge(start) & times.lt(end)]
    if scoped.empty:
        raise ValueError("No records in the selected interval.")
    # Indices refer to the input DataFrame. No complete-hourly assumption.
    return scoped.sort_values(time_column, kind="stable").index.tolist(), start, end

def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen


In [ ]:
def remove_records(
    df, rng, *, time_column, timezone, start_date, end_date,
    key_columns, placement, removal_rate, n_groups, grouped_share,
    expected_frequency=None,
):
    if not key_columns or any(c not in df for c in key_columns):
        raise ValueError("Expected existing key columns.")
    clean = df.reset_index(drop=True).copy()
    if clean[key_columns].isna().any().any() or clean.duplicated(subset=key_columns).any():
        raise ValueError("Expected unique, nonmissing record keys.")
    eligible, start, end = scoped_rows(
        clean, time_column=time_column, timezone=timezone,
        start_date=start_date, end_date=end_date,
    )
    if expected_frequency is not None:
        if key_columns != [time_column]:
            raise ValueError("Expected-frequency check needs one record per timestamp.")
        observed = pd.DatetimeIndex(pd.to_datetime(clean.loc[eligible, time_column], utc=True))
        expected = pd.date_range(start, end, freq=expected_frequency, inclusive="left")
        if not observed.equals(expected):
            raise ValueError("Existing coverage gap or extra record; restore clean data first.")
    selected, _ = select_positions(
        len(eligible), rng, method=placement, rate=removal_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    return clean.drop(index=[eligible[i] for i in selected]).reset_index(drop=True)

In [ ]:
# Small controlled trial (no source file changed).
trial_times = pd.date_range("2025-08-01", periods=7*24, freq="h", tz=TIMEZONE).tz_convert("UTC")
trial = pd.DataFrame({
    "time": trial_times,
    "temperature_2m": [float(10 + i % 17) for i in range(len(trial_times))],
    "relative_humidity_2m": pd.Series([int(40 + i % 40) for i in range(len(trial_times))], dtype="int64"),
    "precipitation": [float(i % 10) for i in range(len(trial_times))],
})

trial_dirty = remove_records(trial, Random(SEED),
    time_column="time", timezone=TIMEZONE, start_date="2025-08-01", end_date="2025-08-08",
    key_columns=["time"], placement="mixed", removal_rate=(0.10, 0.10),
    n_groups=2, grouped_share=0.70, expected_frequency="h")
assert len(trial) - len(trial_dirty) == round(len(trial) * 0.10)
assert not trial_dirty["time"].duplicated().any()
trial.loc[~trial["time"].isin(trial_dirty["time"]), ["time", "temperature_2m"]].head()

In [ ]:
# Verify actual Parquet types and values on a small temporary file.
from tempfile import TemporaryDirectory
with TemporaryDirectory() as temporary:
    clean_path = Path(temporary) / "clean.parquet"
    temp_path = Path(temporary) / "trial.parquet"
    trial.to_parquet(clean_path, index=False)
    trial_dirty.to_parquet(temp_path, index=False)
    roundtrip = pd.read_parquet(temp_path)
    before_schema = pq.read_schema(clean_path)
    after_schema = pq.read_schema(temp_path)
    assert len(roundtrip) == len(trial_dirty)
    assert roundtrip.columns.tolist() == trial_dirty.columns.tolist()
    assert pq.read_metadata(temp_path).num_rows == len(trial_dirty)
    assert all(before_schema.field(c).type == after_schema.field(c).type for c in trial.columns)
    print("Stored Parquet schema:", after_schema)

In [ ]:
# Apply the reusable function to a freshly prepared Parquet artifact.
clean = pd.read_parquet(DATA_PATH)
rng = Random(SEED)
dirty = remove_records(
    clean, rng, time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date=START_DATE, end_date=END_DATE,
    key_columns=KEY_COLUMNS, placement=RECORD_PLACEMENT,
    removal_rate=RECORD_RATE, n_groups=RECORD_GROUPS,
    grouped_share=RECORD_GROUPED_SHARE, expected_frequency=EXPECTED_FREQUENCY,
)
dirty.to_parquet(DATA_PATH, index=False)
saved = pd.read_parquet(DATA_PATH)

In [ ]:
# Inspect: missing expected timestamps; no new nulls are needed to create a gap.
expected = pd.date_range(pd.Timestamp(START_DATE, tz=TIMEZONE).tz_convert("UTC"),
                         pd.Timestamp(END_DATE, tz=TIMEZONE).tz_convert("UTC"),
                         freq=EXPECTED_FREQUENCY, inclusive="left")
observed = pd.DatetimeIndex(pd.to_datetime(saved[TIME_COLUMN], utc=True))
missing = expected.difference(observed)
missing[:15], len(missing)

In [ ]:
# Verify persistence: all changes exist after writing the Parquet file.
assert len(saved) == len(dirty)
assert saved.columns.tolist() == dirty.columns.tolist()
assert pq.read_metadata(DATA_PATH).num_rows == len(saved)
assert len(saved) < len(clean)
assert set(saved[TIME_COLUMN]).issubset(set(clean[TIME_COLUMN]))
print("Removed records:", len(clean) - len(saved))

In [ ]:
# Missing expected records per local date.
dates = pd.Series(missing.tz_convert(TIMEZONE).date)
counts = dates.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 3))
counts.plot.bar(ax=ax)
ax.set(title="Missing hourly records by date", xlabel="Local date", ylabel="Missing records")
plt.tight_layout(); plt.show()